Scenario: Customer Support Chatbot Workflow

Imagine a company wants to build a chatbot that helps customers with quick answers. The workflow is modeled as a graph of states:

- State Definition (BotState)
- The chatbot keeps track of:
- The question asked by the customer.
- The answer generated.
- The history of all past questions.
- Think of this as the chatbot’s notebook where it records the conversation.
- Nodes (Functions)
- get_answer:

When a customer asks, “What are your store hours?”, the chatbot looks at the question and generates a placeholder answer like “Answer to: What are your store hours?”.
It also adds the question to the history log.
- format_output:

Before sending the response back to the customer, the chatbot reformats it into a friendly style:
“Bot says: Answer to: What are your store hours?”
- Graph Flow
- The chatbot starts at the get_answer node (entry point).
- Once the answer is generated, it flows to the format_output node.
- Finally, the conversation ends at END, meaning the chatbot has delivered its response.

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

# 1. Define State
class BotState(TypedDict):
    question: str
    answer: str
    history: list

# 2. Define Nodes (functions)
def get_answer(state: BotState):
    q = state["question"]
    # In real app: call LLM here
    ans = f"Answer to: {q}"
    return {"answer": ans,
            "history": state["history"] + [q]}

def format_output(state: BotState):
    return {"answer": f"Bot says: {state['answer']}"}

# 3. Build the Graph
graph = StateGraph(BotState)
graph.add_node("get_answer", get_answer)
graph.add_node("format", format_output)

# 4. Add Edges
graph.set_entry_point("get_answer")
graph.add_edge("get_answer", "format")
graph.add_edge("format", END)


In [ ]:
# Scenario: Customer Support Chatbot (Question-Based)
# Imagine a company has deployed a chatbot that answers customer
#  questions by calling the Groq API. The workflow is modeled as a
#  graph of states, where each customer query flows through nodes until
#   a response is delivered.

# 1. State Definition
# The chatbot maintains a notebook-like state:
# - question → The customer’s query.
# - answer → The response generated by Groq.
# - history → A log of all past questions.


from langgraph.graph import StateGraph, END
from typing import TypedDict
import requests
from google.colab import userdata

# 1. Define State
class BotState(TypedDict):
    question: str
    answer: str
    history: list

# 2. Define Nodes (functions)
def get_answer(state: BotState):
    q = state["question"]
    groq_api_key = userdata.get('Groq_api')

    if not groq_api_key:
        raise ValueError("Groq API key not found in Colab secrets. Please set 'Groq_api'.")

    # Call Groq API
    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            # IMPORTANT: The list of supported models by Groq API changes frequently.
            # Please refer to the official Groq documentation (https://console.groq.com/docs/models)
            # to find a currently active and supported model name and replace 'YOUR_GROQ_MODEL_NAME_HERE' below.
            # Examples of often available models include 'llama3-8b-8192' or 'llama3-70b-8192',
            # but these can also become decommissioned.
            "model": "llama-3.1-8b-instant",   # Changed to a currently active model
            "messages": [{"role": "user", "content": q}],
        }
    )

    if response.status_code != 200:
        try:
            error_details = response.json()
        except requests.exceptions.JSONDecodeError:
            error_details = response.text
        raise Exception(f"Groq API error (Status: {response.status_code}): {error_details}")

    response_json = response.json()
    if "choices" not in response_json or not response_json["choices"]:
        raise ValueError(f"Unexpected API response format: 'choices' key missing or empty. Full response: {response_json}")

    # Extract answer from Groq response
    ans = response_json["choices"][0]["message"]["content"]

    return {
        "answer": ans,
        "history": state["history"] + [q]
    }

def format_output(state: BotState):
    return {"answer": f"Bot says: {state['answer']}"}

# 3. Build the Graph
graph = StateGraph(BotState)
graph.add_node("get_answer", get_answer)
graph.add_node("format", format_output)

# 4. Add Edges
graph.set_entry_point("get_answer")
graph.add_edge("get_answer", "format")
graph.add_edge("format", END)

# 5. Example Run
if __name__ == "__main__":
    # Initial state
    state = {"question": "What are your store hours?", "answer": "", "history": []}

    # Run the graph
    app = graph.compile()
    result = app.invoke(state)

    print(result["answer"])